In [ ]:
# RQ3b: does verification's extra communication slow responses on slow links? (spec 2, H3b; decision 28: 128 MiB
# x 1 KiB, the loopback baseline and the three dummynet profiles, 30 client queries each)
import json, sys
from pathlib import Path

import pandas as pd

repo = Path.cwd().parent
sys.path.insert(0, str(repo / "src"))
from cs4406_sem_project.analysis import amortise, frame, plots
from cs4406_sem_project.harness import loadgen, session_check
from cs4406_sem_project.harness.grid import fmt_size

RUNS = [repo / "results/raw/all-20261112_tier1", repo / "results/raw/all-20261112_anchors"]  # notebook 17
platform = json.loads((RUNS[0] / "meta.json").read_text())["platform"]
df, status = frame.load_runs(RUNS, platform)
FIG = repo / "docs/figures"
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 400)


def save(name, fig):
    for ext in ("pdf", "png"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=200, bbox_inches="tight")


plots._style()
CELL = (128 << 20, 1 << 10)  # decision 28: the RQ4 cell, 128 MiB x 1 KiB (S3 at 1 bit)

NETEM = repo / "results/raw/p6-20261113/netem"
netem_rows = frame.load_netem(NETEM, platform)
t = plots.build(df, status, netem_rows=netem_rows)
print(f"{len(netem_rows)} client rows;", netem_rows.groupby("profile").size().to_dict())

In [ ]:
# Gate G4 (spec 11, test_netem_calibration): each profile's link as dummynet.sh last recorded it before the run,
# its configured values beside the calibrated ones (decision 22: LAN's RTT is recorded, not checked).
for d in sorted(p for p in NETEM.iterdir() if p.is_dir()):
    state = frame._link_state(d)
    keys = ("profile", "down_mbps", "up_mbps", "rtt_ms", "calibrated", "calib_down_mbps", "calib_up_mbps",
            "calib_rtt_ms", "rtt_checked")
    print(d.name, {k: state.get(k) for k in keys} if state else "loopback, no link")

In [ ]:
# Emulated end-to-end latency (query generation to verified record, over the link) beside the model
# (amortise.latency_e2e from the grid's online medians and the profile), per profile and arm, in seconds.
em = t.emulation.copy()
em["record"] = em.record_bytes.map(lambda r: fmt_size(int(r)) if r else "1bit")
cols = ["profile", "arm", "record", "n", "emulated_p50", "emulated_p95", "modelled_p50", "modelled_p95",
        "emulated_over_modelled", "online_bytes"]
print(em[cols].sort_values(["profile", "arm"]).to_string(index=False, float_format=lambda v: f"{v:.4g}"))

In [ ]:
# H3b: on mobile the latency overhead approaches the byte overhead; on LAN it does not. Per price pair and
# profile, the verifiable arm's emulated p50 latency and online bytes over its baseline's.
eo = plots.emulation_overheads(t.emulation)
print(eo.sort_values(["pair", "profile"]).to_string(index=False, float_format=lambda v: f"{v:.3f}"))

In [ ]:
# F5 (modelled latency per profile at the largest size every record-size arm ran) and F9 (emulated against
# modelled, and latency overhead against byte overhead).
for name in ("F5_network", "F9_emulated_network"):
    save(name, plots.FIGURE_BUILDERS[name](t))
print(sorted(p.name for p in FIG.glob("F[59]_*")))